# Recipe relaxation gates, solve policies, and overrides

Global `RelaxationSettings` describe the solver. A `SolvePolicy`
describes what one recipe gate must achieve, while
`RelaxationOverrides` temporarily changes how that gate approaches its
target. This supports contact-first assembly followed by strict final
bending cleanup.

In [ ]:
# Policies control acceptance; overrides control the path there.
import tangle

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `name` | Stage label used in reports. | string |
| `solver_penetration` | Residual the stage solver works toward. | m |
| `solver_curvature_ratio` | Curvature target used by the stage solver. | ratio |
| `acceptance_penetration` | Residual required to accept the operation. | m |
| `penetration_enforcement` | Whether failure rejects the stage. | `hard` or `soft` |
| `acceptance_curvature_ratio` | Curvature ratio required for acceptance. | ratio |
| `curvature_enforcement` | Whether curvature failure rejects the stage. | `hard` or `soft` |
| `maximum_iterations` | Stage-specific iteration budget. | count |
| `on_exhaustion` | Behavior when the budget is exhausted. | `reject` or `continue` |

In [ ]:
# This assembly stage must resolve contact but temporarily accepts a
# curvature ratio up to 5 so bending cannot block deposition.
contact_first = tangle.SolvePolicy(
    "contact-first settling",
    solver_penetration=0.1e-6,
    solver_curvature_ratio=5.0,
    acceptance_penetration=0.2e-6,
    penetration_enforcement="hard",
    acceptance_curvature_ratio=5.0,
    curvature_enforcement="soft",
    maximum_iterations=10_000,
    on_exhaustion="reject",
)

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `motion_model` | Temporary fiber motion model. | string or `None` |
| `correction_fraction` | Temporary contact correction fraction. | float or `None` |
| `contact_aggregation` | Temporary contact aggregation rule. | string or `None` |
| `stretch_stiffness` | Temporary rest-length stiffness. | float or `None` |
| `bend_stiffness` | Temporary rest-bend stiffness. | float or `None` |
| `curvature_limit_stiffness` | Temporary hard-curvature stiffness. | float or `None` |
| `constraint_iterations` | Temporary constraint sweep count. | integer or `None` |
| `curvature_cleanup_sweeps` | Temporary curvature cleanup count. | integer or `None` |

In [ ]:
# Overrides apply only to relax_with_policy; global settings return for
# later operations.
overrides = tangle.RelaxationOverrides()
overrides.bend_stiffness = 0.1
overrides.curvature_limit_stiffness = 1.0
overrides.contact_aggregation = "deepest_only"

recipe = tangle.Recipe(tangle.Cell([1e-3, 1e-3, 1e-3]))
# These calls represent distinct amounts or acceptance conditions.
recipe.relax_for(100)  # fixed work; no acceptance gate
recipe.relax(maximum_iterations=2_000)  # default hard gate
recipe.relax_with_policy(contact_first, overrides)
recipe.relax_until_targets_reached(0.1e-6, 5_000)
recipe.set_material_bend_radius("fiber", 50e-6)
print(*recipe.operations(), sep="\n")